# MIMIC-IV Continuous IQN Training Pipeline — Local

Local version of the Colab training pipeline. Assumes:
- Working directory: `/home/jay/Uni/SafetyResearch`
- Python venv at `.venv/`
- CSV/NPZ data already in `data/continuous_mimic/`

**Steps:** Preprocess → Train NCDE → Encode → Train IQN

In [ ]:
import os, sys

REPO = os.path.expanduser('~/Uni/SafetyResearch')
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

print("Working directory:", os.getcwd())

## Step 1: Build raw NPZ files from CSVs

Skip if `data/continuous_mimic/reduced_format.npz` already exists.

In [ ]:
import pandas as pd
import numpy as np
import gzip

STATIC_COLS   = ['o:gender', 'o:admission_age', 'o:height', 'o:weight']
TEMPORAL_COLS = [
    'm:timestep',
    'o:heart_rate', 'o:sbp', 'o:dbp', 'o:mbp', 'o:resp_rate', 'o:temperature',
    'o:glucose', 'o:so2', 'o:po2', 'o:pco2', 'o:fio2', 'o:pao2fio2ratio',
    'o:ph', 'o:baseexcess', 'o:chloride', 'o:calcium', 'o:potassium', 'o:sodium',
    'o:lactate', 'o:hematocrit', 'o:hemoglobin', 'o:platelet', 'o:wbc', 'o:albumin',
    'o:aniongap', 'o:bicarbonate', 'o:pt', 'o:ptt', 'o:gcs',
    'o:spo2', 'o:bun', 'o:creatinine', 'o:inr', 'o:bilirubin', 'o:alt', 'o:ast',
    'o:UO', 'o:ventilation'
]
ACTION_COLS   = ['a:action_fluid', 'a:action_vaso']
OUTCOME_COL   = ['r:reward']

def read_csv_auto(path):
    try:
        with gzip.open(path, 'rt') as f:
            return pd.read_csv(f)
    except Exception:
        return pd.read_csv(path)

def csv_to_npz(csv_path, npz_path):
    if os.path.exists(npz_path):
        print(f"  {npz_path} already exists, skipping.")
        return
    print(f"Reading {csv_path} ...")
    df = read_csv_auto(csv_path)
    print(f"  {len(df)} rows, {df['m:stay_id'].nunique()} unique stays")
    static_frame = df[['m:stay_id'] + STATIC_COLS].groupby('m:stay_id', sort=True).first()
    unique_ids = static_frame.index.tolist()
    static_data, temporal_data, action_data, outcome_data = [], [], [], []
    df_by_stay = df.set_index('m:stay_id')
    for sid in unique_ids:
        rows = df_by_stay.loc[[sid]].sort_values('m:timestep')
        static_data.append(static_frame.loc[sid].values.astype(np.float32))
        temporal_data.append(rows[TEMPORAL_COLS].values.astype(np.float32))
        action_data.append(rows[ACTION_COLS].values.astype(np.float32))
        outcome_data.append(rows[OUTCOME_COL].values.astype(np.float32))
    print(f"  Saving to {npz_path} ...")
    np.savez(
        npz_path,
        static_data=np.stack(static_data),
        temporal_data=np.array(temporal_data, dtype=object),
        action_data=np.array(action_data, dtype=object),
        outcome_data=np.array(outcome_data, dtype=object),
        static_columns=STATIC_COLS,
        temporal_columns=TEMPORAL_COLS,
        stay_id=np.array(unique_ids, dtype=np.float64),
    )
    print(f"  Done ({len(unique_ids)} patients).")

DATA_DIR = 'data/continuous_mimic'
csv_to_npz(os.path.join(DATA_DIR, 'final_trajs_noFill.csv'),
           os.path.join(DATA_DIR, 'reduced_format.npz'))
csv_to_npz(os.path.join(DATA_DIR, 'final_trajs_overlapCohort_noFill.csv'),
           os.path.join(DATA_DIR, 'reduced_format_overlapCohort.npz'))
print("NPZ files ready.")

## Step 2: Preprocess — masks, intensities, normalization, rectilinear interpolation

Skip if `data/continuous_mimic/rectilinear_processed/` already exists.

In [ ]:
import subprocess, sys

processed = 'data/continuous_mimic/rectilinear_processed/improved-neural-cdes_data.npz'
if os.path.exists(processed):
    print("Preprocessed data already exists, skipping.")
else:
    result = subprocess.run([sys.executable, 'preprocess_ncde_data.py'],
                            capture_output=False)
    if result.returncode != 0:
        raise RuntimeError("preprocess_ncde_data.py failed")

## Step 3: Train NCDE state encoder

In [ ]:
import torch, yaml, random
from ncde_utils import load_data, evaluator
from ncde import NeuralCDE

NCDE_PARAMS = {
    'hidden_dim':          64,
    'hidden_hidden_dim':   64,
    'pred_num_layers':     3,
    'pred_num_units':      128,
    'lr':                  5e-4,
    'num_training_epochs': 50,
    'lr_step_size':        20,
    'lr_gamma':            0.5,
}
CHECKPOINT_EVERY = 5

config_dict = yaml.safe_load(open('./configs/ncde_config.yaml'))
DATA_DIR   = config_dict['data_dir']
OUTPUT_DIR = config_dict['output_dir']
BATCH_SIZE = config_dict['batch_size']
SEED       = config_dict.get('seed', 2022)

os.makedirs(OUTPUT_DIR, exist_ok=True)
NCDE_CKPT_PATH = os.path.join(OUTPUT_DIR, 'ncde_checkpoint.pt')
NCDE_BEST_PATH = os.path.join(OUTPUT_DIR, 'best_model.pt')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

torch.manual_seed(SEED)
random.seed(SEED)
if device.type == 'cuda':
    torch.cuda.manual_seed(SEED)

(train_loader, val_loader, _), input_dim, action_dim, static_dim, output_dim = load_data(
    data_dir=DATA_DIR, batch_size=BATCH_SIZE
)
(combined_loader, _, _), _, _, _, _ = load_data(
    data_dir=DATA_DIR, batch_size=BATCH_SIZE, combine_train_val=True, shuffle=True
)
print(f"Input dim: {input_dim}, Action dim: {action_dim}, Static dim: {static_dim}, Output dim: {output_dim}")

model = NeuralCDE(
    input_dim, NCDE_PARAMS['hidden_dim'], output_dim, static_dim, action_dim,
    hidden_hidden_dim=NCDE_PARAMS['hidden_hidden_dim'],
    pred_num_layers=NCDE_PARAMS['pred_num_layers'],
    pred_num_units=NCDE_PARAMS['pred_num_units'],
    return_sequences=True, device=device
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=NCDE_PARAMS['lr'], amsgrad=True)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=int(NCDE_PARAMS['lr_step_size']),
    gamma=NCDE_PARAMS['lr_gamma']
)

start_epoch = 0
if os.path.exists(NCDE_CKPT_PATH):
    print(f"Resuming from checkpoint: {NCDE_CKPT_PATH}")
    ckpt = torch.load(NCDE_CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch = ckpt['epoch'] + 1
    print(f"  Resuming from epoch {start_epoch}")
else:
    print("No checkpoint found — training from scratch.")

num_epochs = NCDE_PARAMS['num_training_epochs']
model.train()

for epoch in range(start_epoch, num_epochs):
    loss_pred = 0
    print(f"Epoch {epoch+1}/{num_epochs}", end=" ... ", flush=True)

    for inputs, masks, lengths, targets, _, _ in combined_loader:
        static, temporal, actions = inputs
        static   = static.to(device)
        temporal = temporal.to(device)
        actions  = actions.to(device)
        masks    = masks.to(device)
        lengths  = lengths.to(device)
        targets  = targets.to(device)

        max_length = int(lengths.max().item())
        temporal = temporal[:, :(2*max_length)-1, :]
        actions  = actions[:, :max_length, :]
        targets  = targets[:, :max_length, :]
        masks    = masks[:, :max_length, :-1]

        preds, _ = model((static, temporal, actions))
        loss = model.calculate_loss(preds, targets, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        loss_pred += loss.detach().cpu().item()

    print(f"loss = {loss_pred:.4f}")

    if (epoch + 1) % CHECKPOINT_EVERY == 0 or (epoch + 1) == num_epochs:
        torch.save({
            'epoch':           epoch,
            'model':           model.state_dict(),
            'optimizer':       optimizer.state_dict(),
            'scheduler':       scheduler.state_dict(),
            'hyperparameters': NCDE_PARAMS,
        }, NCDE_CKPT_PATH)
        print(f"  Checkpoint saved (epoch {epoch+1})")

torch.save({'model': model.state_dict(), 'hyperparameters': NCDE_PARAMS}, NCDE_BEST_PATH)
print(f"\nNCDE model saved to: {NCDE_BEST_PATH}")

## Step 4: Encode all trajectories into NCDE hidden states

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, 'encode_data.py'], capture_output=False)
if result.returncode != 0:
    raise RuntimeError('encode_data.py failed')

In [ ]:
import numpy as np

DATA_DIR = 'data/continuous_mimic/rectilinear_processed'
for split in ['train', 'validation', 'test']:
    npz = np.load(os.path.join(DATA_DIR, f'encoded_{split}.npz'), allow_pickle=True)
    s, a, r = npz['states'], npz['actions'], npz['rewards']
    print(f"  {split:12s}: states {s.shape}, actions {a.shape}, rewards {r.shape}")
    print(f"               actions dtype={a.dtype}, range=[{a.min():.3f}, {a.max():.3f}]")

## Step 5: Train continuous IQN (D-network + R-network)

In [ ]:
import subprocess, sys
# Smoke test
result = subprocess.run([sys.executable, 'train_rl.py', '-c', 'iqn_continuous_mimic', '-o', 'smoke_test', 'True'], capture_output=False)
if result.returncode != 0:
    raise RuntimeError('Smoke test failed')

In [ ]:
import subprocess, sys

result = subprocess.run([
    sys.executable, 'train_rl.py', '-c', 'iqn_continuous_mimic',
    '-o', 'num_q_hidden_units', '256',
    '-o', 'num_q_layers', '3',
    '-o', 'num_iqn_samples_train', '64',
    '-o', 'num_iqn_samples_est', '64',
    '-o', 'K_actions', '128',
    '-o', 'use_cql', 'True',
    '-o', 'cql_weight', '0.5',
    '-o', 'train_batch_size', '256',
    '-o', 'num_epochs', '200',
    '-o', 'lr', '0.0001',
    '-o', 'num_ps', '2',
    '-o', 'num_ns', '4',
], capture_output=False)
if result.returncode != 0:
    raise RuntimeError('IQN training failed')

In [ ]:
import glob

ckpt_dir = 'runs/iqn_continuous_mimic'
ckpts = sorted(glob.glob(os.path.join(ckpt_dir, '*.pt')))
for c in ckpts:
    print(f"  {c}  ({os.path.getsize(c)/1e6:.1f} MB)")
if not ckpts:
    print("No checkpoints found.")

## (Optional) Plot training and validation loss curves

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

ckpt_dir = 'runs/iqn_continuous_mimic'
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, sided_Q in zip(axes, ['negative', 'positive']):
    loss_path = os.path.join(ckpt_dir, f'q_losses_{sided_Q}.npy')
    ckpt_path = os.path.join(ckpt_dir, f'q_parameters{sided_Q}.pt')
    if os.path.exists(loss_path):
        ax.plot(np.load(loss_path), label='Train loss')
    if os.path.exists(ckpt_path):
        val_loss = torch.load(ckpt_path, map_location='cpu').get('validation_loss', [])
        if val_loss:
            ax.plot(val_loss, label='Val loss', linestyle='--')
    ax.set_title('D-network' if sided_Q == 'negative' else 'R-network')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Continuous IQN Training Curves — MIMIC-IV Sepsis', fontsize=13)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: training_curves.png")